In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [5]:
model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)

In [4]:
# Create a State
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [6]:
def generate_outline_state(state: BlogState) -> BlogState:
    
    # Fetch Title
    title = state['title']
    
    # Call llm gen outline
    prompt = f'Generate an outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content
    
    # Update state
    state['outline'] = outline
    
    # Return state
    return state

In [7]:
def generate_blog_state(state: BlogState) -> BlogState:
    
    # Fetch Title & Outline
    title = state['title']
    outline = state['outline']
    
    # Call llm gen outline
    prompt = f'Generate a detailed blog on the topic - {title} and outline is - {outline}'
    blog = model.invoke(prompt).content
    
    # Update state
    state['content'] = blog
    
    # Return state
    return state

In [8]:
graph = StateGraph(BlogState)

# Add Nodes
graph.add_node('generate_outline_state', generate_outline_state)
graph.add_node('generate_blog_state', generate_blog_state)

# Add Edges
graph.add_edge(START, 'generate_outline_state')
graph.add_edge('generate_outline_state', 'generate_blog_state')
graph.add_edge('generate_blog_state', END)

workflow = graph.compile()

In [9]:
initial_state = {'title': 'Virat Kohli'}
final_state = workflow.invoke(initial_state)
print("Final State : ", final_state)

Final State :  {'title': 'Virat Kohli', 'outline': 'Here\'s a comprehensive outline for a blog post about Virat Kohli, designed to be engaging, informative, and cover various facets of his career and persona.\n\n---\n\n## Blog Title Options:\n\n*   **Virat Kohli: Decoding the Modern Cricket King**\n*   **The Phenomenon: Unpacking the Impact of Virat Kohli**\n*   **Beyond the Bat: The Unstoppable Rise of Virat Kohli**\n*   **King Kohli: Journey of a Cricketing Legend**\n\n---\n\n## Blog Outline: Virat Kohli\n\n**Target Audience:** Cricket fans, sports enthusiasts, those interested in leadership and success stories.\n**Tone:** Informative, appreciative, slightly awe-inspired, analytical.\n\n---\n\n### I. Introduction (Approx. 150-200 words)\n\n*   **A. Hook:** Start with a powerful statement about Virat Kohli\'s unparalleled impact on modern cricket and his global icon status.\n    *   *Example:* "In the annals of cricket, few names resonate with the same intensity, passion, and sheer do